In [1]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
import torch.optim as optim
import os
from sklearn.decomposition import TruncatedSVD
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# Choice of Weights for each Cluster

<table border="1" cellspacing="0" cellpadding="8" style="width: 100%; border-collapse: collapse;">
  <thead>
    <tr>
      <th style="width: 5%"><strong>Cluster</strong></th>
      <th style="width: 10%"><strong>Listener Type</strong></th>
      <th style="width: 30%"><strong>Characteristics</strong></th>
      <th style="width: 10%"><strong>Model Weights</strong></th>
      <th style="width: 15%"><strong>Feature Weights</strong></th>
      <th style="width: 30%"><strong>Recommendation Strategy</strong></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>Consistent Loyalists</td>
      <td>These users have relatively uniform playlists—likely made up of similar artists, song types, and genre. They're not very exploratory and prefer familiar, consistent music.</td>
      <td>
        SVD: 0.3<br>
        DNN: 0.7
      </td>
      <td>
        Popularity: 0.2<br>
        Era: 0.3<br>
        Length: 0.1<br>
        Sentiment: 0.1<br>
        Genre: 0.3
      </td>
      <td>Prioritize content-based filtering (DNN) with higher weights (0.7) and lower reliance on collaborative filtering (SVD) (0.3). Focus on features like era and genre to ensure recommendations stay closely aligned with established tastes.</td>
    </tr>
    <tr>
      <td>1</td>
      <td>Trend-Savvy Seekers</td>      
      <td>These users explore widely across both popular and obscure tracks, artists, and genres. They're trendy and curious, enjoying playlists that span musical styles and voices.</td>
      <td>
        SVD: 0.6<br>
        DNN: 0.4
      </td>
      <td>
        Popularity: 0.15<br>
        Era: 0.3<br>
        Length: 0.1<br>
        Sentiment: 0.3<br>
        Genre: 0.15
      </td>
      <td>Emphasize collaborative filtering (SVD) with a higher weight (0.6) for capturing broader trends. Encourage exploration in sentiment and era, with the goal of diversifying the user’s musical journey beyond their current preferences.</td>
    </tr>
    <tr>
      <td>2</td>
      <td>Mood-Oriented Focused</td>
      <td>These users craft focused, consistent playlists—likely centered on a particular genre and mood. They're mood-oriented listeners who value emotional or stylistic cohesion.</td>
      <td>
        SVD: 0.2<br>
        DNN: 0.8
      </td>
      <td>
        Popularity: 0.1<br>
        Era: 0.1<br>
        Length: 0.1<br>
        Sentiment: 0.35<br>
        Genre: 0.35
      </td>
      <td>Favor content-based filtering (DNN) with a higher weight (0.8) to ensure mood consistency. Focus on sentiment and genre to maintain a high relevance score, while reducing the emphasis on popularity, era, and length.</td>
    </tr>
    <tr>
      <td>3</td>
      <td>Universal Explorers</td>
      <td>These users seek diverse musical experiences. They enjoy mixing eras, moods, and artists, crafting emotionally varied and eclectic playlists.</td>
      <td>
        SVD: 0.8<br>
        DNN: 0.2
      </td>
      <td>
        Popularity: 0.4<br>
        Era: 0.05<br>
        Length: 0.05<br>
        Sentiment: 0.1<br>
        Genre: 0.4
      </td>
      <td>Prioritize collaborative filtering (SVD) with a higher weight (0.8) to tap into broader listening patterns. Focus on discovering a wider variety of music, especially in popularity and genre, while maintaining some diversity across other features.</td>
    </tr>
  </tbody>
</table>

In [2]:
# Global default feature weights for DNN (can be overridden per cluster)
default_weights = {
    'popularity': 0.2,
    'era': 0.2,
    'length': 0.2,
    'sentiment': 0.2,
    'genre': 0.2
}

# Hybrid combination weights (can also be modified per cluster if needed)
weight_svd = 0.5
weight_dnn = 0.5

# Define hybrid weights per cluster (example values)
cluster_hybrid_weights = {
    0: {'svd': 0.3, 'dnn': 0.7},
    1: {'svd': 0.6, 'dnn': 0.4},
    2: {'svd': 0.2, 'dnn': 0.8},
    3: {'svd': 0.8, 'dnn': 0.2},
}

# For demonstration, we define a dictionary of cluster-specific weights.
# Modify these as necessary.
cluster_weights = {
    0: {'popularity': 0.2, 'era': 0.3, 'length': 0.1, 'sentiment': 0.1, 'genre': 0.3},
    1: {'popularity': 0.15, 'era': 0.3, 'length': 0.1, 'sentiment': 0.3, 'genre': 0.15},
    2: {'popularity': 0.1, 'era': 0.1, 'length': 0.1, 'sentiment': 0.35, 'genre': 0.35},
    3: {'popularity': 0.4, 'era': 0.05, 'length': 0.05, 'sentiment': 0.1, 'genre': 0.4},
}

In [ ]:
# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== GLOBAL CONFIG ==========
DATA_PATH = ""
PLAYLIST_FILE = os.path.join(DATA_PATH, "playlist_final_unweighted.csv")
TRACKS_FILE = os.path.join(DATA_PATH, "track_final.parquet")
NUM_PLAYLISTS = 1000000000  # Adjust as needed
K_EVAL = 50

# ========== SVD PARAMETERS ==========
LATENT_DIM = 100

# ========== DNN PARAMETERS ==========
HIDDEN_SIZES = [512, 256, 128]
DROPOUT_RATE = 0.1
ACTIVATION = nn.ReLU
BATCH_SIZE = 512
EPOCHS = 30
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
LR_STEP_SIZE = 5
LR_GAMMA = 0.5
NEGATIVE_SAMPLE_RATIO = 10
POS_WEIGHT_MULTIPLIER = 1.5

# Collaborative Filtering using SVD

### Model Loading & Pre-Processing

In [4]:
# ==========================================================
# 1. DATA LOADING & PREPROCESSING (SVD & DNN)
# ==========================================================

def load_data():
    # Load the full dataset (no cluster filtering for SVD)
    playlists = pd.read_csv(PLAYLIST_FILE, engine='python', on_bad_lines='skip')
    tracks = pd.read_parquet(TRACKS_FILE)
    playlists = playlists[:NUM_PLAYLISTS]
    return playlists, tracks

def clean_centroid_string(s):
    if isinstance(s, str):
        s = re.sub(r'[\[\]]', '', s).strip()
        return [float(x) for x in s.split()]
    return s

def parse_track_list(s):
    return [int(x) for x in re.findall(r'\d+', s)]

def preprocess_playlists(df):
    df = df.copy()
    for col in ['sentiment_centroid', 'genre_centroid']:
        if col in df.columns:
            df[col] = df[col].apply(clean_centroid_string)
    for col in ['track_idx_list', 'tracks_to_predict']:
        if col in df.columns:
            df[col] = df[col].apply(parse_track_list)
    return df

def split_data(playlists):
    # Split based on the 'dataset_type' column.
    train = playlists[playlists['dataset_type'].isin(['train', 'val'])].reset_index(drop=True)
    val = playlists[playlists['dataset_type'] == 'val'].reset_index(drop=True)
    test = playlists[playlists['dataset_type'] == 'test'].reset_index(drop=True)
    final = playlists[playlists['dataset_type'] == 'final'].reset_index(drop=True)
    return train, val, test, final

# Build track mapping and interaction matrix for SVD
def build_track_mapping(tracks_df):
    unique_tracks = sorted(tracks_df['track_idx'].unique().tolist())
    track_to_col = {track: idx for idx, track in enumerate(unique_tracks)}
    return unique_tracks, track_to_col

def build_interaction_matrix(playlists_subset, track_to_col, track_col='track_idx_list'):
    n = len(playlists_subset)
    m = len(track_to_col)
    matrix = np.zeros((n, m), dtype=np.int32)
    for i, row in playlists_subset.iterrows():
        for track in row[track_col]:
            if track in track_to_col:
                matrix[i, track_to_col[track]] = 1
    return matrix

def create_track_feat_map_svd(tracks_df):
    feat_cols = ['track_popularity',
                 'Early Years', 'Classic Era', 'Golden Era', '2000s', 'Modern Era',
                 'Short', 'Medium', 'Long',
                 'joy', 'calm', 'sadness', 'fear', 'energizing', 'dreamy',
                 'Instrumental / Ambient Sounds', 'Soft Acoustic / Classical', 'Orchestral / Soundtrack',
                 'Mid-tempo Pop / Indie', 'Upbeat Electronic / Dance', 'Slow & Melancholic (Sad Songs)',
                 'Experimental / Jazz Fusion', 'Lo-Fi / Chill Vibes']
    feat_map = {row['track_idx']: row[feat_cols].values for _, row in tracks_df.iterrows()}
    return feat_map

In [5]:
# ==========================================================
# 2. SVD MODEL TRAINING, PREDICTION & EVALUATION (Global)
# ==========================================================

def train_svd_model(interaction_matrix_train):
    svd = TruncatedSVD(n_components=LATENT_DIM, random_state=42)
    U_train = svd.fit_transform(interaction_matrix_train)
    Sigma = svd.singular_values_
    VT = svd.components_
    sqrt_sigma = np.sqrt(Sigma)
    P_train = U_train * sqrt_sigma
    Q = (VT.T * sqrt_sigma)
    return svd, P_train, Q, sqrt_sigma

def fold_in_playlists(svd, interaction_matrix, sqrt_sigma):
    return svd.transform(interaction_matrix) * sqrt_sigma

def predict_mf(playlist_latent, interaction_row, Q):
    scores = playlist_latent.dot(Q.T)
    existing = np.where(interaction_row > 0)[0]
    scores[existing] = -np.inf
    return scores

def predict_mf_wrapper(playlist_idx, interaction_matrix, P_matrix, Q):
    return predict_mf(P_matrix[playlist_idx], interaction_matrix[playlist_idx], Q)

def get_all_svd_predictions(interaction_matrix, P_matrix, predict_func, Q, test_playlists):
    predictions = {}
    n = interaction_matrix.shape[0]
    for i in range(n):
        playlist_id = test_playlists.iloc[i]['playlist_idx']
        predictions[playlist_id] = predict_func(i, interaction_matrix, P_matrix, Q)
    return predictions

def build_ground_truth(playlists_subset, track_to_col, track_col='tracks_to_predict'):
    test_items = {}
    for _, row in playlists_subset.iterrows():
        pid = row['playlist_idx']
        test_items[pid] = [track_to_col[t] for t in row[track_col] if t in track_to_col]
    return test_items

In [6]:
# --------------------------
# Run SVD on the full dataset
# --------------------------
playlist_raw, tracks = load_data()
playlists = preprocess_playlists(playlist_raw)
train_playlists, val_playlists, test_playlists, final_playlists = split_data(playlists)
unique_tracks, track_to_col = build_track_mapping(tracks)
print("Training playlists:", len(train_playlists))
print("Unique tracks:", len(unique_tracks))

interaction_matrix_train = build_interaction_matrix(train_playlists, track_to_col)
interaction_matrix_test = build_interaction_matrix(test_playlists, track_to_col)
print("Training matrix shape:", interaction_matrix_train.shape)
print("Test matrix shape:", interaction_matrix_test.shape)

track_feat_map_svd = create_track_feat_map_svd(tracks)

svd, P_train, Q, sqrt_sigma = train_svd_model(interaction_matrix_train)
P_test = fold_in_playlists(svd, interaction_matrix_test, sqrt_sigma)
svd_predictions = get_all_svd_predictions(interaction_matrix_test, P_test, predict_mf_wrapper, Q, test_playlists)

Training playlists: 13877
Unique tracks: 252236
Training matrix shape: (13877, 252236)
Test matrix shape: (1851, 252236)


# Content Based Filtering using DNN

### Model Loading & Pre-Processing

In [7]:
###################################
# 1. DATA LOADING & PREPROCESSING (DNN)
###################################
def preprocess_playlist(playlist):
    # Convert centroids from strings to numpy arrays and expand them
    playlist['sentiment_centroid'] = playlist['sentiment_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))
    playlist['genre_centroid'] = playlist['genre_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))
    sent_df = playlist['sentiment_centroid'].apply(pd.Series)
    sent_df.columns = [f'sent{i+1}' for i in range(sent_df.shape[1])]
    genre_df = playlist['genre_centroid'].apply(pd.Series)
    genre_df.columns = [f'genre{i+1}' for i in range(genre_df.shape[1])]
    playlist = pd.concat([playlist.drop(columns=['sentiment_centroid', 'genre_centroid']), sent_df, genre_df], axis=1)
    def convert_string_array_to_list(s):
        if isinstance(s, str):
            return [int(x) for x in re.findall(r'\d+', s)]
        return []
    playlist['track_idx_list'] = playlist['track_idx_list'].apply(convert_string_array_to_list)
    playlist['tracks_to_predict'] = playlist['tracks_to_predict'].apply(convert_string_array_to_list)
    cols = ['playlist_idx', 'dataset_type', 'track_idx_list', 'tracks_to_predict', 'cluster',
            'popularity_mean', 'era_early_years_proportion', 'era_classic_era_proportion', 'era_golden_era_proportion',
            'era_2000s_proportion', 'era_modern_era_proportion', 'length_short_proportion', 'length_medium_proportion',
            'length_long_proportion', 'sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6',
            'genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']
    return playlist[cols]

# Use the same load_data() as before
playlist_raw_dnn, tracks_dnn = load_data()
playlist_dnn = preprocess_playlist(playlist_raw_dnn)
train_playlists_dnn, val_playlists_dnn, test_playlists_dnn, final_playlists_dnn = split_data(playlist_dnn)

# Define playlist feature columns for DNN
playlist_popularity_cols = ['popularity_mean']
playlist_era_cols = ['era_early_years_proportion', 'era_classic_era_proportion', 'era_golden_era_proportion',
                       'era_2000s_proportion', 'era_modern_era_proportion']
playlist_length_cols = ['length_short_proportion', 'length_medium_proportion', 'length_long_proportion']
playlist_sentiment_cols = ['sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6']
playlist_genre_cols = ['genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']

playlist_scalers = {
    'pop_playlist': StandardScaler(),
    'era_playlist': StandardScaler(),
    'len_playlist': StandardScaler(),
    'sent_playlist': StandardScaler(),
    'genre_playlist': StandardScaler()
}

playlist_scalers['pop_playlist'].fit(train_playlists_dnn[playlist_popularity_cols])
playlist_scalers['era_playlist'].fit(train_playlists_dnn[playlist_era_cols])
playlist_scalers['len_playlist'].fit(train_playlists_dnn[playlist_length_cols])
playlist_scalers['sent_playlist'].fit(train_playlists_dnn[playlist_sentiment_cols])
playlist_scalers['genre_playlist'].fit(train_playlists_dnn[playlist_genre_cols])

def normalize_playlist_features(df):
    df[playlist_popularity_cols] = playlist_scalers['pop_playlist'].transform(df[playlist_popularity_cols])
    df[playlist_era_cols] = playlist_scalers['era_playlist'].transform(df[playlist_era_cols])
    df[playlist_length_cols] = playlist_scalers['len_playlist'].transform(df[playlist_length_cols])
    df[playlist_sentiment_cols] = playlist_scalers['sent_playlist'].transform(df[playlist_sentiment_cols])
    df[playlist_genre_cols] = playlist_scalers['genre_playlist'].transform(df[playlist_genre_cols])
    return df

train_playlists_dnn = normalize_playlist_features(train_playlists_dnn.copy())
val_playlists_dnn = normalize_playlist_features(val_playlists_dnn.copy())
test_playlists_dnn = normalize_playlist_features(test_playlists_dnn.copy())

def create_playlist_feat_map(df):
    feat_cols = playlist_popularity_cols + playlist_era_cols + playlist_length_cols + playlist_sentiment_cols + playlist_genre_cols
    feat_map = {row['playlist_idx']: row[feat_cols].values for _, row in df.iterrows()}
    return feat_map

playlist_feat_map_train = create_playlist_feat_map(train_playlists_dnn)
playlist_feat_map_test = create_playlist_feat_map(test_playlists_dnn)

# For tracks in DNN, define feature columns and scalers
popularity_cols = ['track_popularity']
era_cols = ['Early Years', 'Classic Era', 'Golden Era', '2000s', 'Modern Era']
length_cols = ['Short', 'Medium', 'Long']
sentiment_cols = ['joy', 'calm', 'sadness', 'fear', 'energizing', 'dreamy']
genre_cols = ['Instrumental / Ambient Sounds', 'Soft Acoustic / Classical', 'Orchestral / Soundtrack',
              'Mid-tempo Pop / Indie', 'Upbeat Electronic / Dance', 'Slow & Melancholic (Sad Songs)',
              'Experimental / Jazz Fusion', 'Lo-Fi / Chill Vibes']

track_scalers = {
    'pop_track': StandardScaler(),
    'era_track': StandardScaler(),
    'len_track': StandardScaler(),
    'sent_track': StandardScaler(),
    'genre_track': StandardScaler()
}

track_scalers['pop_track'].fit(tracks_dnn[popularity_cols])
track_scalers['era_track'].fit(tracks_dnn[era_cols])
track_scalers['len_track'].fit(tracks_dnn[length_cols])
track_scalers['sent_track'].fit(tracks_dnn[sentiment_cols])
track_scalers['genre_track'].fit(tracks_dnn[genre_cols])

def normalize_track_features(df):
    df[popularity_cols] = track_scalers['pop_track'].transform(df[popularity_cols])
    df[era_cols] = track_scalers['era_track'].transform(df[era_cols])
    df[length_cols] = track_scalers['len_track'].transform(df[length_cols])
    df[sentiment_cols] = track_scalers['sent_track'].transform(df[sentiment_cols])
    df[genre_cols] = track_scalers['genre_track'].transform(df[genre_cols])
    return df

tracks_dnn = normalize_track_features(tracks_dnn.copy())

def create_track_feat_map(df):
    feat_cols = popularity_cols + era_cols + length_cols + sentiment_cols + genre_cols
    feat_map = {int(row['track_idx']): row[feat_cols].values for _, row in df.iterrows()}
    return feat_map

track_feat_map = create_track_feat_map(tracks_dnn)

def apply_weights(features, weights):
    split_sizes = [1, 5, 3, 6, 8]
    chunks = np.split(features, np.cumsum(split_sizes)[:-1])
    return np.concatenate([chunk * weights[key] for chunk, key in zip(chunks, weights)])

In [8]:
###################################
# 2. MODEL & DATASET DEFINITION (DNN)
###################################
class PlaylistTrackDataset(Dataset):
    def __init__(self, playlist_df, feat_map, track_map, n_neg=NEGATIVE_SAMPLE_RATIO, cluster_weights=None):
        self.samples = []
        self.track_map = track_map
        all_tids = list(track_map.keys())
        cluster_weights = cluster_weights or {}
        for _, row in playlist_df.iterrows():
            pid, cluster = row['playlist_idx'], row['cluster']
            pos_tracks = row['tracks_to_predict']
            if not pos_tracks:
                continue
            p_feat = feat_map[pid]
            weights = cluster_weights.get(cluster, default_weights)
            p_feat_w = apply_weights(p_feat, weights)
            for tid in pos_tracks:
                if tid in track_map:
                    self.samples.append((p_feat_w, track_map[tid], 1))
            negs = np.random.choice(list(set(all_tids) - set(pos_tracks)),
                                    min(len(pos_tracks) * n_neg, len(all_tids)), replace=False)
            for tid in negs:
                self.samples.append((p_feat_w, track_map[tid], 0))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        p, t, y = self.samples[idx]
        p = np.array(p, dtype=np.float32).flatten()
        t = np.array(t, dtype=np.float32).flatten()
        return torch.tensor(np.concatenate([p, t]), dtype=torch.float32), torch.tensor([y], dtype=torch.float32)

class DNNRecommender(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        layers = []
        prev_size = input_size
        for hidden in HIDDEN_SIZES:
            layers.append(nn.Linear(prev_size, hidden))
            layers.append(ACTIVATION())
            layers.append(nn.BatchNorm1d(hidden))
            layers.append(nn.Dropout(DROPOUT_RATE))
            prev_size = hidden
        layers.append(nn.Linear(prev_size, 1))
        self.model = nn.Sequential(*layers)
    def forward(self, x):
        return self.model(x).squeeze()

def predict_dnn(playlist_idx, feat_map, track_feat_map, model):
    p_feat = feat_map[playlist_idx]
    p_feat_w = apply_weights(p_feat, default_weights)
    all_track_ids = list(track_feat_map.keys())
    p_feat_tensor = torch.tensor(np.array(p_feat_w, dtype=np.float32)).repeat(len(all_track_ids), 1).to(device)
    t_feats = np.array([track_feat_map[tid] for tid in all_track_ids], dtype=np.float32)
    t_tensor = torch.tensor(t_feats, dtype=torch.float32).to(device)
    inputs = torch.cat([p_feat_tensor, t_tensor], dim=1)
    model.eval()
    with torch.no_grad():
        scores = torch.sigmoid(model(inputs)).cpu().numpy().flatten()
    return scores

def get_all_dnn_predictions(feat_map, track_feat_map, test_playlists, model):
    predictions = {}
    for i, row in test_playlists.iterrows():
        playlist_id = row['playlist_idx']
        predictions[playlist_id] = predict_dnn(playlist_id, feat_map, track_feat_map, model)
    return predictions

# Cluster-Specific DNN Training & Hybrid Combination

In [9]:
# ==========================================================
# 5. CLUSTER-SPECIFIC DNN TRAINING & HYBRID COMBINATION
# ==========================================================
all_cluster_recs = []  # To collect recommendations for each cluster
sorted_dnn_track_ids = sorted(track_feat_map.keys())
trained_cluster_models = {}  # Global dict to store trained models

def train_cluster(cluster):
    print(f"\n=== Processing Cluster {cluster} ===")

    # Filter data for the given cluster
    cluster_train_dnn = train_playlists_dnn[train_playlists_dnn['cluster'] == cluster].reset_index(drop=True)
    cluster_test_dnn = test_playlists_dnn[test_playlists_dnn['cluster'] == cluster].reset_index(drop=True)

    if cluster_train_dnn.empty or cluster_test_dnn.empty:
        print(f"Skipping cluster {cluster} due to insufficient data.")
        return None

    current_weights = cluster_weights.get(cluster, default_weights)

    cluster_feat_map_train = create_playlist_feat_map(cluster_train_dnn)
    cluster_feat_map_test = create_playlist_feat_map(cluster_test_dnn)

    cluster_train_dataset = PlaylistTrackDataset(
        cluster_train_dnn,
        cluster_feat_map_train,
        track_feat_map,
        n_neg=NEGATIVE_SAMPLE_RATIO,
        cluster_weights={cluster: current_weights}
    )
    cluster_train_loader = DataLoader(cluster_train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    cluster_model = DNNRecommender(input_size=len(next(iter(cluster_train_dataset))[0])).to(device)
    cluster_optimizer = optim.AdamW(cluster_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    cluster_scheduler = torch.optim.lr_scheduler.StepLR(cluster_optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)

    pos = sum(1 for _, _, l in cluster_train_dataset.samples if l == 1)
    neg = sum(1 for _, _, l in cluster_train_dataset.samples if l == 0)
    cluster_pos_weight_val = (neg / pos * POS_WEIGHT_MULTIPLIER) if pos > 0 else 1.0
    cluster_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(cluster_pos_weight_val).to(device))

    print(f"Training DNN for cluster {cluster} with {len(cluster_train_dataset)} samples...")

    # Epoch training loop
    for epoch in range(EPOCHS):
        cluster_model.train()
        total_loss = 0.0
        for xb, yb in cluster_train_loader:
            xb, yb = xb.to(device), yb.to(device)
            cluster_optimizer.zero_grad()
            loss = cluster_criterion(cluster_model(xb).view(-1), yb.view(-1))
            loss.backward()
            cluster_optimizer.step()
            total_loss += loss.item()
        cluster_scheduler.step()
        print(f"Cluster {cluster}, Epoch {epoch+1}, Loss: {total_loss:.4f}")

    # DNN Predictions
    cluster_dnn_predictions = get_all_dnn_predictions(cluster_feat_map_test, track_feat_map, cluster_test_dnn, cluster_model)

    # --- Optimized Hybrid Combination ---
    cluster_track_ids = set()
    for tracks in cluster_train_dnn['track_idx_list']:
        cluster_track_ids.update(tracks)
    union_track_ids_cluster = sorted(cluster_track_ids)
    union_track_to_index_cluster = {tid: idx for idx, tid in enumerate(union_track_ids_cluster)}
    union_length_cluster = len(union_track_ids_cluster)

    # Precompute once per cluster
    dnn_track_list_cluster = [tid for tid in sorted_dnn_track_ids if tid in union_track_to_index_cluster]
    dnn_map = {tid: idx for idx, tid in enumerate(dnn_track_list_cluster)}

    svd_track_indices_cluster = {
        track: idx for track, idx in track_to_col.items() if track in union_track_to_index_cluster
    }

    current_hybrid = cluster_hybrid_weights.get(cluster, {'svd': weight_svd, 'dnn': weight_dnn})
    cluster_final_predictions = {}

    for pid in tqdm(cluster_test_dnn['playlist_idx'], desc=f'Hybrid predictions cluster {cluster}'):
        final_score = np.zeros(union_length_cluster, dtype=np.float32)

        if pid in svd_predictions:
            svd_vec = svd_predictions[pid]
            svd_indices, svd_scores = zip(*[
                (union_track_to_index_cluster[track], svd_vec[idx])
                for track, idx in svd_track_indices_cluster.items()
            ])
            final_score[list(svd_indices)] += current_hybrid['svd'] * np.array(svd_scores)

        if pid in cluster_dnn_predictions:
            dnn_vec = cluster_dnn_predictions[pid]
            valid_len = min(len(dnn_track_list_cluster), len(dnn_vec))
            dnn_indices = [union_track_to_index_cluster[track] for track in dnn_track_list_cluster[:valid_len]]
            final_score[dnn_indices] += current_hybrid['dnn'] * dnn_vec[:valid_len]

        cluster_final_predictions[pid] = final_score

    cluster_recommendations = []
    for pid, final_score in cluster_final_predictions.items():
        ranked_indices = np.argsort(-final_score)[:K_EVAL]
        recommended_tracks = [union_track_ids_cluster[i] for i in ranked_indices]
        cluster_recommendations.append({'playlist_idx': pid, 'hybrid_recommendations': recommended_tracks})

    cluster_rec_df = pd.DataFrame(cluster_recommendations)

    print(f"\nCluster {cluster} final hybrid recommendations:")
    print(cluster_rec_df.head())

    # Save the trained model for reuse later.
    trained_cluster_models[cluster] = cluster_model

    return cluster_rec_df

In [10]:
cluster0_recs = train_cluster(0)
all_cluster_recs.append(cluster0_recs)


=== Processing Cluster 0 ===
Training DNN for cluster 0 with 210760 samples...
Cluster 0, Epoch 1, Loss: 497.1355
Cluster 0, Epoch 2, Loss: 452.7574
Cluster 0, Epoch 3, Loss: 435.2501
Cluster 0, Epoch 4, Loss: 423.7844
Cluster 0, Epoch 5, Loss: 415.2610
Cluster 0, Epoch 6, Loss: 395.7586
Cluster 0, Epoch 7, Loss: 389.4925
Cluster 0, Epoch 8, Loss: 382.4213
Cluster 0, Epoch 9, Loss: 375.2487
Cluster 0, Epoch 10, Loss: 369.8877
Cluster 0, Epoch 11, Loss: 355.0153
Cluster 0, Epoch 12, Loss: 346.3793
Cluster 0, Epoch 13, Loss: 341.6380
Cluster 0, Epoch 14, Loss: 338.5366
Cluster 0, Epoch 15, Loss: 333.6354
Cluster 0, Epoch 16, Loss: 323.4195
Cluster 0, Epoch 17, Loss: 320.4976
Cluster 0, Epoch 18, Loss: 316.3300
Cluster 0, Epoch 19, Loss: 314.3519
Cluster 0, Epoch 20, Loss: 311.4785
Cluster 0, Epoch 21, Loss: 305.9458
Cluster 0, Epoch 22, Loss: 301.4779
Cluster 0, Epoch 23, Loss: 303.1639
Cluster 0, Epoch 24, Loss: 300.8512
Cluster 0, Epoch 25, Loss: 300.3748
Cluster 0, Epoch 26, Loss: 29

Hybrid predictions cluster 0: 100%|██████████| 255/255 [00:27<00:00,  9.24it/s]



Cluster 0 final hybrid recommendations:
   playlist_idx                             hybrid_recommendations
0            10  [208248, 94251, 122853, 218307, 221104, 123705...
1            31  [245418, 225394, 32793, 30534, 83087, 251981, ...
2            46  [111112, 148576, 203885, 43926, 218361, 114006...
3            56  [191313, 232829, 33445, 34319, 87806, 154321, ...
4           126  [224606, 80649, 90560, 194274, 226723, 248435,...


In [11]:
cluster1_recs = train_cluster(1)
all_cluster_recs.append(cluster1_recs)


=== Processing Cluster 1 ===
Training DNN for cluster 1 with 506660 samples...
Cluster 1, Epoch 1, Loss: 1130.3582
Cluster 1, Epoch 2, Loss: 1035.8872
Cluster 1, Epoch 3, Loss: 999.2126
Cluster 1, Epoch 4, Loss: 977.5576
Cluster 1, Epoch 5, Loss: 960.2118
Cluster 1, Epoch 6, Loss: 924.2643
Cluster 1, Epoch 7, Loss: 910.5261
Cluster 1, Epoch 8, Loss: 900.5114
Cluster 1, Epoch 9, Loss: 888.8750
Cluster 1, Epoch 10, Loss: 880.9020
Cluster 1, Epoch 11, Loss: 852.7561
Cluster 1, Epoch 12, Loss: 842.5549
Cluster 1, Epoch 13, Loss: 836.8588
Cluster 1, Epoch 14, Loss: 829.7304
Cluster 1, Epoch 15, Loss: 822.2522
Cluster 1, Epoch 16, Loss: 807.0955
Cluster 1, Epoch 17, Loss: 801.3725
Cluster 1, Epoch 18, Loss: 795.6412
Cluster 1, Epoch 19, Loss: 794.9350
Cluster 1, Epoch 20, Loss: 786.6469
Cluster 1, Epoch 21, Loss: 777.9373
Cluster 1, Epoch 22, Loss: 775.3984
Cluster 1, Epoch 23, Loss: 773.2649
Cluster 1, Epoch 24, Loss: 770.9645
Cluster 1, Epoch 25, Loss: 770.1029
Cluster 1, Epoch 26, Loss: 

Hybrid predictions cluster 1: 100%|██████████| 614/614 [02:02<00:00,  5.01it/s]



Cluster 1 final hybrid recommendations:
   playlist_idx                             hybrid_recommendations
0             7  [191313, 111863, 216260, 163571, 89133, 237774...
1           121  [163571, 249870, 244627, 47657, 154321, 172218...
2           165  [27407, 229720, 86790, 209437, 124094, 191177,...
3           179  [139020, 244723, 33216, 99015, 128945, 250245,...
4           198  [249785, 6666, 191177, 35237, 26659, 250306, 1...


In [12]:
cluster2_recs = train_cluster(2)
all_cluster_recs.append(cluster2_recs)


=== Processing Cluster 2 ===
Training DNN for cluster 2 with 426800 samples...
Cluster 2, Epoch 1, Loss: 822.7492
Cluster 2, Epoch 2, Loss: 739.4704
Cluster 2, Epoch 3, Loss: 701.9566
Cluster 2, Epoch 4, Loss: 680.9570
Cluster 2, Epoch 5, Loss: 662.5594
Cluster 2, Epoch 6, Loss: 628.7797
Cluster 2, Epoch 7, Loss: 613.4328
Cluster 2, Epoch 8, Loss: 600.9202
Cluster 2, Epoch 9, Loss: 592.3031
Cluster 2, Epoch 10, Loss: 584.0481
Cluster 2, Epoch 11, Loss: 559.1036
Cluster 2, Epoch 12, Loss: 552.1660
Cluster 2, Epoch 13, Loss: 542.4456
Cluster 2, Epoch 14, Loss: 537.4646
Cluster 2, Epoch 15, Loss: 532.7721
Cluster 2, Epoch 16, Loss: 516.6712
Cluster 2, Epoch 17, Loss: 511.4337
Cluster 2, Epoch 18, Loss: 507.0946
Cluster 2, Epoch 19, Loss: 506.9276
Cluster 2, Epoch 20, Loss: 501.0307
Cluster 2, Epoch 21, Loss: 493.4892
Cluster 2, Epoch 22, Loss: 490.7433
Cluster 2, Epoch 23, Loss: 488.5961
Cluster 2, Epoch 24, Loss: 487.1045
Cluster 2, Epoch 25, Loss: 485.3696
Cluster 2, Epoch 26, Loss: 48

Hybrid predictions cluster 2: 100%|██████████| 518/518 [01:15<00:00,  6.84it/s]



Cluster 2 final hybrid recommendations:
   playlist_idx                             hybrid_recommendations
0            17  [235367, 237415, 216374, 64550, 14975, 54047, ...
1            28  [250306, 26659, 57996, 86790, 167186, 56156, 6...
2            29  [235367, 237495, 250306, 86790, 103098, 203557...
3            72  [86790, 86135, 64550, 237495, 193787, 221079, ...
4            85  [235367, 221079, 116049, 203557, 86790, 237415...


In [13]:
cluster3_recs = train_cluster(3)
all_cluster_recs.append(cluster3_recs)


=== Processing Cluster 3 ===
Training DNN for cluster 3 with 382250 samples...
Cluster 3, Epoch 1, Loss: 928.8804
Cluster 3, Epoch 2, Loss: 866.6449
Cluster 3, Epoch 3, Loss: 844.8193
Cluster 3, Epoch 4, Loss: 829.3675
Cluster 3, Epoch 5, Loss: 815.9263
Cluster 3, Epoch 6, Loss: 790.5465
Cluster 3, Epoch 7, Loss: 778.7795
Cluster 3, Epoch 8, Loss: 769.2812
Cluster 3, Epoch 9, Loss: 761.2226
Cluster 3, Epoch 10, Loss: 753.4833
Cluster 3, Epoch 11, Loss: 731.4622
Cluster 3, Epoch 12, Loss: 723.3214
Cluster 3, Epoch 13, Loss: 718.0331
Cluster 3, Epoch 14, Loss: 712.1724
Cluster 3, Epoch 15, Loss: 705.4133
Cluster 3, Epoch 16, Loss: 692.4328
Cluster 3, Epoch 17, Loss: 686.8747
Cluster 3, Epoch 18, Loss: 680.2744
Cluster 3, Epoch 19, Loss: 677.7123
Cluster 3, Epoch 20, Loss: 676.3983
Cluster 3, Epoch 21, Loss: 665.4182
Cluster 3, Epoch 22, Loss: 661.7992
Cluster 3, Epoch 23, Loss: 662.3563
Cluster 3, Epoch 24, Loss: 658.7936
Cluster 3, Epoch 25, Loss: 656.6386
Cluster 3, Epoch 26, Loss: 65

Hybrid predictions cluster 3: 100%|██████████| 464/464 [02:00<00:00,  3.84it/s]



Cluster 3 final hybrid recommendations:
   playlist_idx                             hybrid_recommendations
0             2  [174997, 186858, 175642, 111887, 157693, 19963...
1            47  [219637, 131992, 77604, 249070, 48713, 232953,...
2            60  [76387, 135471, 135948, 157693, 232845, 130032...
3            64  [208248, 94251, 121753, 181349, 122853, 224428...
4           108  [246233, 34094, 122597, 56322, 192501, 83411, ...


In [14]:
# Combine recommendations from all clusters into one DataFrame
if all_cluster_recs:
    combined_rec_df = pd.concat(all_cluster_recs, ignore_index=True)
    print("\nCombined Hybrid Recommendations:")
    print(combined_rec_df.head())
    # combined_rec_df.to_csv('combined_rec_df.csv', index=False)
else:
    print("No cluster recommendations generated.")


Combined Hybrid Recommendations:
   playlist_idx                             hybrid_recommendations
0            10  [208248, 94251, 122853, 218307, 221104, 123705...
1            31  [245418, 225394, 32793, 30534, 83087, 251981, ...
2            46  [111112, 148576, 203885, 43926, 218361, 114006...
3            56  [191313, 232829, 33445, 34319, 87806, 154321, ...
4           126  [224606, 80649, 90560, 194274, 226723, 248435,...


# Validation Set Performance

In [23]:
# Read the original playlist and tracks files (which contain cluster information in the playlists)
tracks = pd.read_parquet(TRACKS_FILE)
playlist_original = pd.read_csv(PLAYLIST_FILE, engine='python', on_bad_lines='skip')

# Prepare a DataFrame with playlist_idx and tracks_to_predict
playlist_temp = pd.read_csv(PLAYLIST_FILE, engine='python', on_bad_lines='skip')
playlist_temp = playlist_temp[['playlist_idx', 'tracks_to_predict']]

# Merge recommendations with ground truth from playlists
playlist_recommendation = playlist_temp.merge(combined_rec_df, on='playlist_idx', how='inner')

In [24]:
# Convert string representations of arrays to lists
def convert_string_array_to_list(s):
    if isinstance(s, str):
        numbers = re.findall(r'\d+', s)
        return [int(x) for x in numbers]
    elif isinstance(s, np.ndarray):
        return s.astype(int).tolist()
    elif isinstance(s, list):
        return [int(x) for x in s if str(x).isdigit()]
    return []

playlist_recommendation['tracks_to_predict'] = playlist_recommendation['tracks_to_predict'].apply(convert_string_array_to_list)
playlist_recommendation['hybrid_recommendations'] = playlist_recommendation['hybrid_recommendations'].apply(convert_string_array_to_list)

In [25]:
# Adjust tracks and playlist_original for evaluation
tracks['track_popularity'] = tracks['track_popularity'] / 100
tracks['track_idx'] = tracks['track_idx'].astype(int)
tracks = tracks.set_index('track_idx')
playlist_original['popularity_mean'] = playlist_original['popularity_mean'] / 100
playlist_original['playlist_idx'] = playlist_original['playlist_idx'].astype(int)
# For evaluation purposes, we need cluster information.
# Merge the cluster column from playlist_original into the recommendation DataFrame.
playlist_recommendation = playlist_recommendation.merge(
    playlist_original[['playlist_idx', 'cluster']],
    on='playlist_idx',
    how='left'
)

In [26]:
# Define the track feature columns for evaluation
tracks_relevant_columns = tracks[['track_popularity',
                                  'Early Years', 'Classic Era', 'Golden Era', '2000s', 'Modern Era',
                                  'Short', 'Medium', 'Long',
                                  "joy", "calm", "sadness", "fear", "energizing", "dreamy",
                                  "Instrumental / Ambient Sounds", "Soft Acoustic / Classical", "Orchestral / Soundtrack",
                                  "Mid-tempo Pop / Indie", "Upbeat Electronic / Dance", "Slow & Melancholic (Sad Songs)",
                                  "Experimental / Jazz Fusion", "Lo-Fi / Chill Vibes"]]

In [27]:
playlist_original['sentiment_centroid'] = playlist_original['sentiment_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))
playlist_original['genre_centroid'] = playlist_original['genre_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))

# Unpack the column into separate columns
playlist_original[['genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']] = pd.DataFrame(playlist_original['genre_centroid'].tolist())

# Drop the original column if you no longer need it
playlist_original = playlist_original.drop(columns=['genre_centroid'])

# Unpack the column into separate columns
playlist_original[['sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6']] = pd.DataFrame(playlist_original['sentiment_centroid'].tolist())

# Drop the original column if you no longer need it
playlist_original = playlist_original.drop(columns=['sentiment_centroid'])

playlist_relevant_columns = playlist_original[['popularity_mean',
                                      'era_early_years_proportion', 'era_classic_era_proportion', 'era_golden_era_proportion', 'era_2000s_proportion', 'era_modern_era_proportion',
                                      'length_short_proportion', 'length_medium_proportion', 'length_long_proportion',
                                      'sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6',
                                      'genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']]

playlist_original = playlist_original.set_index('playlist_idx')

In [28]:
# --- Define Evaluation Functions ---

def avg_cosine_similarity(set_a, set_b, tracks_df, track_feat_columns):
    set_a = list(set_a)
    set_b = list(set_b)
    try:
        a_vectors = tracks_df.loc[set_a, track_feat_columns].values
        b_vectors = tracks_df.loc[set_b, track_feat_columns].values
    except KeyError:
        return 0.0
    if a_vectors.shape[0] == 0 or b_vectors.shape[0] == 0:
        return 0.0
    sim_matrix = cosine_similarity(a_vectors, b_vectors)
    return sim_matrix.mean()

def compute_relevance_for_playlist(row, tracks_df, track_feat_columns):
    true_tracks = row['tracks_to_predict']
    recommended_tracks = row['hybrid_recommendations']
    sim_score = avg_cosine_similarity(recommended_tracks, true_tracks, tracks_df, track_feat_columns)
    return sim_score

def calculate_diversity_score(recommended_tracks, tracks_df, track_feat_columns):
    try:
        recommended_vectors = tracks_df.loc[recommended_tracks, track_feat_columns].values
    except KeyError:
        return 0.0
    if recommended_vectors.shape[0] == 0:
        return 0.0
    sim_matrix = cosine_similarity(recommended_vectors)
    np.fill_diagonal(sim_matrix, 0)
    avg_similarity = sim_matrix.sum() / (sim_matrix.shape[0] * (sim_matrix.shape[0] - 1))
    diversity_score = 1 - avg_similarity
    return diversity_score

def compute_diversity_for_playlist(row, tracks_df, track_feat_columns):
    recommended_tracks = row['hybrid_recommendations']
    diversity_score = calculate_diversity_score(recommended_tracks, tracks_df, track_feat_columns)
    return diversity_score

def compute_novelty(row, playlist_relevant_df, tracks_df, playlist_feat_columns, track_feat_columns):
    try:
        playlist_vector = playlist_relevant_df.loc[row.name, playlist_feat_columns].values.reshape(1, -1)
    except KeyError:
        return 0.0
    recommended_ids = list(row['hybrid_recommendations'])
    try:
        track_vectors = tracks_df.loc[recommended_ids, track_feat_columns].values
    except KeyError:
        return 0.0
    if track_vectors.shape[0] == 0:
        return 0.0
    sim_scores = cosine_similarity(playlist_vector, track_vectors)[0]
    novelty_score = 1 - np.mean(sim_scores)
    return novelty_score

In [29]:
# --- Compute Evaluation Metrics Per Cluster ---

# First, reset indices so that we can group by cluster easily.
playlist_recommendation = playlist_recommendation.reset_index(drop=True)

# Create a dictionary to store metrics for each cluster.
cluster_results = {}

# Get unique clusters from the recommendations.
clusters = sorted(playlist_recommendation['cluster'].unique())

for clus in clusters:
    print(f"\nEvaluating Cluster: {clus}")
    cluster_df = playlist_recommendation[playlist_recommendation['cluster'] == clus].copy()

    # Compute relevance, diversity, novelty per playlist in this cluster.
    cluster_df['relevance'] = cluster_df.apply(lambda row: compute_relevance_for_playlist(row, tracks, tracks_relevant_columns.columns), axis=1)
    cluster_df['diversity'] = cluster_df.apply(lambda row: compute_diversity_for_playlist(row, tracks, tracks_relevant_columns.columns), axis=1)
    cluster_df['novelty'] = cluster_df.apply(lambda row: compute_novelty(row, playlist_original, tracks, playlist_relevant_columns.columns, tracks_relevant_columns.columns), axis=1)
    cluster_df['serendipity'] = cluster_df['relevance'] * cluster_df['novelty']

    # Aggregate the metrics for this cluster.
    cluster_results[clus] = {
        'mean_relevance': cluster_df['relevance'].mean(),
        'mean_diversity': cluster_df['diversity'].mean(),
        'mean_novelty': cluster_df['novelty'].mean(),
        'mean_serendipity': cluster_df['serendipity'].mean(),
        'n_playlists': len(cluster_df)
    }

    print(f"Cluster {clus} Metrics:")
    print(f"  Mean Relevance: {cluster_results[clus]['mean_relevance']:.4f}")
    print(f"  Mean Diversity: {cluster_results[clus]['mean_diversity']:.4f}")
    print(f"  Mean Serendipity: {cluster_results[clus]['mean_serendipity']:.4f}")
    print(f"  Number of Playlists: {cluster_results[clus]['n_playlists']}")

# Optionally, you can convert the results into a DataFrame:
cluster_metrics_df = pd.DataFrame.from_dict(cluster_results, orient='index')
print("\nCluster Evaluation Metrics:")
print(cluster_metrics_df)


Evaluating Cluster: 0
Cluster 0 Metrics:
  Mean Relevance: 0.6139
  Mean Diversity: 0.3669
  Mean Serendipity: 0.1587
  Number of Playlists: 255

Evaluating Cluster: 1
Cluster 1 Metrics:
  Mean Relevance: 0.6237
  Mean Diversity: 0.3438
  Mean Serendipity: 0.1548
  Number of Playlists: 614

Evaluating Cluster: 2
Cluster 2 Metrics:
  Mean Relevance: 0.6293
  Mean Diversity: 0.3496
  Mean Serendipity: 0.1593
  Number of Playlists: 518

Evaluating Cluster: 3
Cluster 3 Metrics:
  Mean Relevance: 0.4855
  Mean Diversity: 0.4604
  Mean Serendipity: 0.1789
  Number of Playlists: 464

Cluster Evaluation Metrics:
   mean_relevance  mean_diversity  mean_novelty  mean_serendipity  n_playlists
0        0.613895        0.366871      0.268058          0.158701          255
1        0.623662        0.343830      0.252285          0.154849          614
2        0.629315        0.349618      0.255470          0.159300          518
3        0.485545        0.460444      0.381599          0.178889      

# Generate Results for Final Test Set

In [30]:
# Run SVD on the full final dataset
interaction_matrix_final = build_interaction_matrix(final_playlists, track_to_col)
print("Training matrix shape:", interaction_matrix_train.shape)
print("Final matrix shape:", interaction_matrix_final.shape)
P_final = fold_in_playlists(svd, interaction_matrix_final, sqrt_sigma)
svd_predictions = get_all_svd_predictions(interaction_matrix_final, P_final, predict_mf_wrapper, Q, final_playlists)

Training matrix shape: (13877, 252236)
Final matrix shape: (2776, 252236)


In [31]:
# -----------------------------------------------------------
# CLUSTER-SPECIFIC DNN TRAINING & HYBRID COMBINATION for Final Test Set
# -----------------------------------------------------------
# Normalize final playlists for DNN predictions and create feature map
final_playlists_dnn = normalize_playlist_features(final_playlists_dnn.copy())
playlist_feat_map_final = create_playlist_feat_map(final_playlists_dnn)
sorted_dnn_track_ids = sorted(track_feat_map.keys())  # Precomputed global sorted track IDs

# Use the same clusters as from training
clusters = sorted(train_playlists_dnn['cluster'].unique())
all_cluster_recs_final = []

for cluster in clusters:
    print(f"\n=== Processing Final for Cluster {cluster} ===")

    # Filter the final DNN data for this cluster
    cluster_final_dnn = final_playlists_dnn[final_playlists_dnn['cluster'] == cluster].reset_index(drop=True)
    if cluster_final_dnn.empty:
        print(f"Skipping cluster {cluster} due to insufficient final data.")
        continue

    # Retrieve the pre-trained model for this cluster
    if cluster not in trained_cluster_models:
        print(f"No trained model available for cluster {cluster}. Skipping.")
        continue
    cluster_model = trained_cluster_models[cluster]

    # Create the feature map for the final set for this cluster
    cluster_feat_map_final = create_playlist_feat_map(cluster_final_dnn)

    # Get DNN predictions using the pre-trained model
    cluster_dnn_predictions = get_all_dnn_predictions(cluster_feat_map_final, track_feat_map, cluster_final_dnn, cluster_model)

    # --- Optimized Hybrid Combination for Final Set ---
    # Derive cluster-specific track IDs from the training playlists
    cluster_train_subset = train_playlists_dnn[train_playlists_dnn['cluster'] == cluster]
    cluster_track_ids = set()
    for tracks in cluster_train_subset['track_idx_list']:
        cluster_track_ids.update(tracks)
    union_track_ids_cluster = sorted(cluster_track_ids)
    union_track_to_index_cluster = {tid: idx for idx, tid in enumerate(union_track_ids_cluster)}
    union_length_cluster = len(union_track_ids_cluster)

    # Precompute the DNN track list for this cluster using global sorted IDs
    dnn_track_list_cluster = [tid for tid in sorted_dnn_track_ids if tid in union_track_to_index_cluster]

    # Precompute SVD indices for tracks in this cluster
    svd_track_indices_cluster = {
        track: idx for track, idx in track_to_col.items() if track in union_track_to_index_cluster
    }

    current_hybrid = cluster_hybrid_weights.get(cluster, {'svd': weight_svd, 'dnn': weight_dnn})
    cluster_final_predictions = {}

    from tqdm import tqdm
    for pid in tqdm(cluster_final_dnn['playlist_idx'], desc=f'Hybrid predictions final cluster {cluster}'):
        final_score = np.zeros(union_length_cluster, dtype=np.float32)

        # Incorporate SVD predictions
        if pid in svd_predictions:
            svd_vec = svd_predictions[pid]
            svd_idx_and_score = [(union_track_to_index_cluster[track], svd_vec[idx])
                                 for track, idx in svd_track_indices_cluster.items()]
            if svd_idx_and_score:
                svd_indices, svd_scores = zip(*svd_idx_and_score)
                final_score[list(svd_indices)] += current_hybrid['svd'] * np.array(svd_scores)

        # Incorporate DNN predictions
        if pid in cluster_dnn_predictions:
            dnn_vec = cluster_dnn_predictions[pid]
            valid_len = min(len(dnn_track_list_cluster), len(dnn_vec))
            dnn_indices = [union_track_to_index_cluster[tid] for tid in dnn_track_list_cluster[:valid_len]]
            final_score[dnn_indices] += current_hybrid['dnn'] * dnn_vec[:valid_len]

        cluster_final_predictions[pid] = final_score

    # Convert hybrid predictions into a recommendations DataFrame
    cluster_recommendations = []
    for pid, final_score in cluster_final_predictions.items():
        ranked_indices = np.argsort(-final_score)[:K_EVAL]
        recommended_tracks = [union_track_ids_cluster[i] for i in ranked_indices]
        cluster_recommendations.append({'playlist_idx': pid, 'hybrid_recommendations': recommended_tracks})

    cluster_rec_df = pd.DataFrame(cluster_recommendations)
    print(f"\nCluster {cluster} final hybrid recommendations:")
    print(cluster_rec_df.head())
    all_cluster_recs_final.append(cluster_rec_df)


=== Processing Final for Cluster 0 ===


Hybrid predictions final cluster 0: 100%|██████████| 383/383 [00:42<00:00,  8.91it/s]



Cluster 0 final hybrid recommendations:
   playlist_idx                             hybrid_recommendations
0             5  [93035, 85382, 14481, 27612, 234902, 155865, 2...
1            16  [140314, 251689, 14975, 17731, 65300, 184725, ...
2            93  [224606, 41929, 194274, 147150, 80649, 39039, ...
3           113  [111332, 199099, 222280, 219594, 155865, 12683...
4           120  [32793, 5923, 167934, 134300, 73413, 85508, 19...

=== Processing Final for Cluster 1 ===


Hybrid predictions final cluster 1: 100%|██████████| 921/921 [03:04<00:00,  4.99it/s]



Cluster 1 final hybrid recommendations:
   playlist_idx                             hybrid_recommendations
0            19  [250666, 232729, 118317, 121753, 237495, 20673...
1            23  [122336, 35237, 202106, 103098, 104413, 102026...
2            77  [118317, 20673, 228264, 201211, 180201, 104659...
3            99  [242528, 168606, 249109, 99481, 237495, 235367...
4           154  [251617, 212881, 235251, 232829, 83460, 250210...

=== Processing Final for Cluster 2 ===


Hybrid predictions final cluster 2: 100%|██████████| 776/776 [01:55<00:00,  6.73it/s]



Cluster 2 final hybrid recommendations:
   playlist_idx                             hybrid_recommendations
0            15  [203557, 250306, 221079, 216374, 140314, 86790...
1            40  [212881, 172218, 61398, 232829, 219637, 232953...
2            57  [140314, 216374, 221079, 86790, 72522, 219734,...
3            62  [14975, 235367, 221079, 216374, 203557, 219734...
4            79  [34052, 146458, 222741, 234725, 99649, 190384,...

=== Processing Final for Cluster 3 ===


Hybrid predictions final cluster 3: 100%|██████████| 696/696 [03:06<00:00,  3.73it/s]



Cluster 3 final hybrid recommendations:
   playlist_idx                             hybrid_recommendations
0            25  [189533, 83411, 246233, 87806, 38245, 183688, ...
1            32  [212881, 172218, 115704, 61398, 35659, 85574, ...
2            38  [212881, 115704, 61398, 172218, 232829, 202567...
3            50  [135471, 84350, 76387, 157693, 168341, 232845,...
4            89  [232281, 81339, 129996, 250306, 27407, 86135, ...


# Save Recommendations

In [32]:
# Combine recommendations from all clusters into one DataFrame
if all_cluster_recs_final:
    final_rec_df = pd.concat(all_cluster_recs_final, ignore_index=True).sort_values(by='playlist_idx', ascending=True)
    output_file_local = "final_recommendations.csv"
    final_rec_df.to_csv(output_file_local, index=False)
    print(f"Final recommendations saved locally as {output_file_local}.")

    # Optionally, export to Google Drive and trigger local download
    from google.colab import drive
    drive.mount('/content/drive')
    output_file_drive = "/content/drive/MyDrive/BT4222 Project/Colab Notebook/final_recommendations.csv"
    final_rec_df.to_csv(output_file_drive, index=False)
    print(f"Final recommendations saved to Google Drive at {output_file_drive}.")

    from google.colab import files
    files.download(output_file_local)
    print("Downloading final_recommendations.csv to your local machine.")
else:
    print("No final recommendations generated.")

Final recommendations saved locally as final_recommendations.csv.
Mounted at /content/drive
Final recommendations saved to Google Drive at /content/drive/MyDrive/BT4222 Project/Colab Notebook/final_recommendations.csv.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>